# дообучение двухбашенной модели

берем предобученный bi энкодер и учим его на наших парах. лосс InfoNCE с in-batch негативами)) , то есть sampled softmax, где негативы для каждого запроса это позитивы остальных запросов в батче. веса башен общие, размораживаем все, кроме матрицы токенов

большой батч нужен для хороших негативов, а в память влезает мало. попробовал применить GradCache: считаем эмбеддинги всего батча без графа, потом пересчитываем маленькими кусками с градиентом

In [ ]:
import sys, time, random
sys.path.insert(0, "..")

from pathlib import Path
import numpy as np
import polars as pl
import torch
from datasets import Dataset
from sentence_transformers import SentenceTransformer, SentenceTransformerTrainer, SentenceTransformerTrainingArguments, losses
from sentence_transformers.training_args import BatchSamplers

from src.text import item_text, query_text
from src.metrics import recall_at_k, recall_by_slice
from src.search import encode, topk

PROC = Path("../data/processed")
MODELS_DIR = Path("../data/models")
MODELS_DIR.mkdir(exist_ok=True)

MODELS = {"user-bge-m3": dict(name="deepvk/USER-bge-m3", q="", p=""), "bge-m3": dict(name="BAAI/bge-m3", q="", p=""), "e5-base": dict(name="intfloat/multilingual-e5-base", q="query: ", p="passage: "), "e5-large-instruct": dict(name="intfloat/multilingual-e5-large-instruct", q="Instruct: Given a search query, retrieve service ads that match it\nQuery: ", p="")}
MODEL_KEY = "user-bge-m3"
RUN_NAME = MODEL_KEY + "-ft1"

QUERY_LEN = 32 # запросы короткие, 32 токенов хватает с запасом
ITEM_LEN = 160 # заголовок + вид/тип + услуги + кусок описания
BATCH = 256 # эффективный батч, столько негативов видит каждый запрос
MINI_BATCH = 16 # сколько реально влезает в память за один проход
LR = 2e-5
TARGET_HOURS = 2.5 # сколько готовы ждать одну эпоху. число пар подберем под это

random.seed(0)
torch.manual_seed(0)
cfg = MODELS[MODEL_KEY]

/Users/amirhamidullin/PycharmProjects/avito_recsys/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/var/folders/lz/_v3mf0j971z3l63wg5q45y8c0000gn/T/ipykernel_3994/4231437145.py:9: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers import SentenceTransformer, SentenceTransformerTrainer, SentenceTransformerTrainingArguments, losses
/var/folders/lz/_v3mf0j971z3l63wg5q45y8c0000gn/T/ipykernel_3994/4231437145.py:10: DeprecationWarning: Importing from 'sentence_transformers.training_args' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.training_args' instead.
  from sentence_transformers.training_args import BatchSamplers


## Пары для обучения

Из трейна без валидационных запросов. текст запроса и объявления собираем теми же шаблонами, что при поиске иначе модель на обучении и на инференсе увидит разное

In [ ]:
train = pl.read_parquet(PROC / "train_rest.parquet", columns=["search_query", "search_infm_params_text", "item_id", "item_title_raw", "item_infm_params_text", "item_description_raw"])
print("строк в трейне", len(train))

# одну и ту же пару текстов учить дважды смысла нет
train = train.unique(subset=["search_query", "search_infm_params_text", "item_id"], maintain_order=True)   # порядок фиксируем иначе выборка пар не воспроизводится
print("уникальных пар", len(train))

t = time.time()
anchors = [cfg["q"] + query_text(q, p) for q, p in zip(train["search_query"], train["search_infm_params_text"])]
positives = [cfg["p"] + item_text(ti, pa, de) for ti, pa, de in zip(train["item_title_raw"], train["item_infm_params_text"], train["item_description_raw"])]
print("тексты собраны за", round((time.time() - t) / 60, 1), "минут")
print("запрос", anchors[0])
print("объявление", positives[0][:250])

строк в трейне: 494646
уникальных пар: 459581


тексты собраны за 0.5 минут

пример пары:
  запрос: щечки яблочки инъекция
  объявление: ботокс лица. вид услуги: красота, здоровье. тип услуги: косметология. услуги: инъекции ботокса в брови, инъекции ботокса вокруг глаз, инъекции ботокса в лоб. 🔈акция до 31 мая ⏳❤️‍🔥 ✨подари своей коже молодость и сияние✨ 🌝 устали от возрастных изменен


## Модель

Матрицу токенов замораживаем: в xlm-r это четверть миллиона строк по 1024, почти половина всех параметров, а учить ее на нашем объеме незачем 



In [ ]:

model = SentenceTransformer(cfg["name"], device="mps", config_kwargs={"attention_probs_dropout_prob": 0.0})
model.max_seq_length = ITEM_LEN

emb = model[0].auto_model.embeddings.word_embeddings
emb.weight.requires_grad_(False)
total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("параметров всего", round(total / 1e6), "млн, обучаемых", round(trainable / 1e6), "млн")


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 8350.76it/s]

параметров всего: 359 млн, обучаемых: 312 млн


## подсчет времени

короткий прогон на нескольких шагах чтобы понять скорость и выбрать, сколько пар взять в эпоху

In [ ]:
def make_args(out_dir, max_steps=-1, epochs=1):
    return SentenceTransformerTrainingArguments(output_dir=str(out_dir), num_train_epochs=epochs, max_steps=max_steps, per_device_train_batch_size=BATCH, learning_rate=LR, warmup_ratio=0.05, lr_scheduler_type="linear", batch_sampler=BatchSamplers.NO_DUPLICATES, dataloader_drop_last=True, logging_steps=10, save_strategy="steps", save_steps=40, save_total_limit=1, report_to="none", seed=0)

probe_n = BATCH * 3
probe_ds = Dataset.from_dict({"anchor": anchors[:probe_n], "positive": positives[:probe_n]})
loss = losses.CachedMultipleNegativesRankingLoss(model, scale=20.0, mini_batch_size=MINI_BATCH)

t = time.time()
SentenceTransformerTrainer(model=model, args=make_args(MODELS_DIR / "probe", max_steps=3), train_dataset=probe_ds, loss=loss).train()
sec_per_step = (time.time() - t) / 3
pairs_per_hour = BATCH / sec_per_step * 3600
print("секунд на шаг из", BATCH, "пар:", round(sec_per_step, 1))
print("пар в час:", round(pairs_per_hour))
N_TRAIN = int(min(len(anchors), pairs_per_hour * TARGET_HOURS))
N_TRAIN = N_TRAIN - N_TRAIN % BATCH
print("берем в эпоху", N_TRAIN, "пар из", len(anchors), "— это примерно", round(N_TRAIN / pairs_per_hour, 1), "часа")

The `warmup_ratio` argument is deprecated in Transformers v5+, and will also be removed from Sentence Transformers once support for Transformers v4 is dropped. Since you're using Transformers v5+, please use `warmup_steps` (as a float) to specify the warmup ratio instead.



Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

/Users/amirhamidullin/PycharmProjects/avito_recsys/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


секунд на шаг из 256 пар: 18.1
пар в час: 51058
берем в эпоху 127488 пар из 459581 — это примерно 2.5 часа


## Обучение

Перемешиваем пары и берем столько, сколько влезает в бюджет времени. Одна эпоха: это дообучение а не обучение, и обучаемых эмбеддингов по id у нас нет, так что переобучаться после первой эпохи нечему

In [5]:
order = list(range(len(anchors)))
random.shuffle(order)
order = order[:N_TRAIN]
train_ds = Dataset.from_dict({"anchor": [anchors[i] for i in order], "positive": [positives[i] for i in order]})

t = time.time()
trainer = SentenceTransformerTrainer(model=model, args=make_args(MODELS_DIR / RUN_NAME), train_dataset=train_ds, loss=loss)
trainer.train()
print("обучение заняло", round((time.time() - t) / 3600, 2), "часа")

model.save(str(MODELS_DIR / RUN_NAME))
print("модель сохранена в", MODELS_DIR / RUN_NAME)

The `warmup_ratio` argument is deprecated in Transformers v5+, and will also be removed from Sentence Transformers once support for Transformers v4 is dropped. Since you're using Transformers v5+, please use `warmup_steps` (as a float) to specify the warmup ratio instead.


Step,Training Loss
10,1.368980
20,1.222473
30,1.112328
40,1.057575
50,1.058937
60,1.031222
70,1.015702
80,0.942535
90,0.960500
100,0.925536



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.46it/s]


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.45it/s]

...

обучение заняло 3.77 часа



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.22it/s]


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.22it/s]

модель сохранена в ../data/models/user-bge-m3-ft1


## проверка


In [6]:
val = pl.read_parquet(PROC / "val_queries.parquet")
corpus = pl.read_parquet(PROC / "val_corpus.parquet")
item_ids = corpus["item_id"].to_list()
relevant = {q: set(r) for q, r in zip(val["query_id"], val["relevant"])}
is_seen = dict(zip(val["query_id"], val["is_seen"]))

sub_idx = np.load(PROC / "val_corpus_sub_idx.npy")   # то же подмножество, что во втором блокноте
sub_ids = [item_ids[j] for j in sub_idx]
print("подмножество корпуса:", len(sub_idx))

corpus_texts = [item_text(t, p, d) for t, p, d in zip(corpus["item_title_raw"], corpus["item_infm_params_text"], corpus["item_description_raw"])]
query_texts = [query_text(q, p) for q, p in zip(val["search_query"], val["search_infm_params_text"])]

def evaluate(corpus_emb, query_emb, label):
    idx = topk(query_emb, corpus_emb, k=200)
    pred = {q: [sub_ids[j] for j in row] for q, row in zip(val["query_id"], idx)}
    r50 = recall_at_k(pred, relevant, 50)
    by = recall_by_slice(pred, relevant, is_seen, 50)
    print(label, "-> recall@50:", round(r50, 4), " recall@200:", round(recall_at_k(pred, relevant, 200), 4), " (unseen", round(by[False][0], 3), " seen", round(by[True][0], 3), ")")
    return idx

# до обучения, из кэша
zs_path = PROC / "emb" / MODEL_KEY
if (zs_path / "val_corpus.npy").exists():
    zs_corpus = np.load(zs_path / "val_corpus_sub.npy")
    zs_query = np.load(zs_path / "val_queries.npy")
    _ = evaluate(zs_corpus, zs_query, "до обучения")

# после
model.eval()
model.half()
ft_corpus = encode(model, [corpus_texts[j] for j in sub_idx], batch_size=64, prefix=cfg["p"])
ft_query = encode(model, query_texts, batch_size=64, prefix=cfg["q"])
ft_idx = evaluate(ft_corpus, ft_query, "после обучения")

out = PROC / "emb" / RUN_NAME
out.mkdir(parents=True, exist_ok=True)
np.save(out / "val_corpus_sub.npy", ft_corpus)
np.save(out / "val_corpus_sub_idx.npy", sub_idx)
np.save(out / "val_queries.npy", ft_query)

подмножество корпуса: 47864



Batches:   0%|          | 0/748 [00:00<?, ?it/s]


Batches:   0%|          | 1/748 [00:02<25:25,  2.04s/it]


Batches:   0%|          | 2/748 [00:03<19:28,  1.57s/it]


Batches:   0%|          | 3/748 [00:04<17:16,  1.39s/it]

...


Batches:  85%|████████▌ | 34/40 [00:03<00:00, 11.50it/s]


Batches:  90%|█████████ | 36/40 [00:03<00:00, 11.76it/s]


Batches:  95%|█████████▌| 38/40 [00:04<00:00, 12.22it/s]


Batches: 100%|██████████| 40/40 [00:04<00:00,  9.75it/s]

после обучения -> recall@50: 0.648  recall@200: 0.8651  (unseen 0.667  seen 0.616 )
